# get-children-callable-param — worked example 2: Recursive parameters() with dotted names

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `get-children-callable-param`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

`parameters()` is built on top of `get_children` (extended to yield both tensors and sub-modules). It does a depth-first walk: leaf tensors are yielded directly, while `Module`-valued children are recursed into with their names prefixed by the parent attribute, producing the canonical `fc1.weight` state-dict naming.

## Worked solution

We implement the recursive parameter walker.

1. `get_children` now yields both `MiniTensor` and `Module` attributes. `parameters` dispatches on type.
2. For each `(name, val)`: if it is a `MiniTensor`, it is a leaf — `yield name, val` unchanged.
3. If it is a `Module`, recurse: iterate `val.parameters()` and for each `(sub_name, sub_val)` it yields, prefix the name with `f'{name}.{sub_name}'`. This builds the dotted path one level at a time.
4. The walk is depth-first and in-order, so a two-layer MLP yields `fc1.weight, fc1.bias, fc2.weight, fc2.bias` in exactly that order.
5. We construct a small MLP and print the produced names to confirm the dotted convention.

In [ ]:
import torch as t

t.manual_seed(1)

class MiniTensor:
    def __init__(self, data):
        self.data = data

class Module:
    def get_children(self):
        for name, val in self.__dict__.items():
            if isinstance(val, (MiniTensor, Module)):
                yield name, val

    def parameters(self):
        for name, val in self.get_children():
            if isinstance(val, MiniTensor):
                yield name, val
            elif isinstance(val, Module):
                for sub_name, sub_val in val.parameters():
                    yield f'{name}.{sub_name}', sub_val

class Linear(Module):
    def __init__(self, i, o):
        self.weight = MiniTensor(t.randn(o, i))
        self.bias = MiniTensor(t.zeros(o))

class MLP(Module):
    def __init__(self):
        self.fc1 = Linear(3, 5)
        self.fc2 = Linear(5, 2)

names = [n for n, _ in MLP().parameters()]
print(names)
print('dotted:', names == ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'])